## text-speciofic analysis 

Generating basic info of `.rft` documents separated by month and year. Generated data is in `data.json` file.

In [8]:
import os
from striprtf.striprtf import rtf_to_text
from classla import Pipeline
import classla
import json
import re
import pandas as pd
#import nltk

In [ ]:
#nltk.download("punkt")
classla.download("sl")
nlp = Pipeline("sl", processors="tokenize")
path = "../Data/"
subfolders_year = [f.path for f in os.scandir(path) if f.is_dir()]

2025-04-21 00:26:54 INFO: Downloading these customized packages for language: sl (Slovenian)...
| Processor | Package  |
------------------------
| tokenize  | standard |
| pos       | standard |
| lemma     | standard |
| depparse  | standard |
| ner       | standard |
| pretrain  | standard |

2025-04-21 00:26:55 INFO: File exists: C:\Users\lukag\classla_resources\sl\pos\standard.pt.
2025-04-21 00:26:55 INFO: File exists: C:\Users\lukag\classla_resources\sl\lemma\standard.pt.
2025-04-21 00:26:55 INFO: File exists: C:\Users\lukag\classla_resources\sl\depparse\standard.pt.
2025-04-21 00:26:55 INFO: File exists: C:\Users\lukag\classla_resources\sl\ner\standard.pt.
2025-04-21 00:26:55 INFO: File exists: C:\Users\lukag\classla_resources\sl\pretrain\standard.pt.
2025-04-21 00:26:55 INFO: Finished downloading models and saved to C:\Users\lukag\classla_resources.
2025-04-21 00:26:55 INFO: Loading these models for language: sl (Slovenian):
| Processor | Package  |
------------------------
| t

In [ ]:
document_dict = {}
date_pattern = r"(\d{1,2})\. (\d{1,2})\. (\d{4})"

for folder_year in subfolders_year:
    folder_year_name = os.path.basename(folder_year)
    subfolders_month = [f.path for f in os.scandir(folder_year) if f.is_dir()]
    document_dict[folder_year_name] = {
        "months": len(subfolders_month),
        "sentence_count": 0,
        "word_count": 0,
        "months_data": {}
    }
    for folder_month in subfolders_month:
        folder_month_name = os.path.basename(folder_month)
        files = [f for f in os.listdir(folder_month)]
        document_dict[folder_year_name]["months_data"][folder_month_name] = {
            "files": len(files),
            "sentence_count": 0,
            "word_count": 0,
            "file_data": {}
        }

        for file in files:
            with open(os.path.join(folder_month, file), "r", encoding="utf-8") as f:
                file_name = os.path.basename(file)
                rtf_content = f.read()
                plain_text = rtf_to_text(rtf_content)
                plain_text = re.sub(r"\s+", " ", plain_text).strip()
                plain_text = plain_text.replace("\u0000", "")
                modified_text = re.sub(date_pattern, r"\1[PERIOD] \2[PERIOD] \3", plain_text)

                # Tokenize plain text into sentences
                #sentences = nltk.tokenize.sent_tokenize(plain_text, language="sl")
                doc = nlp(modified_text)
                sentences = [sentence.text for sentence in doc.sentences]

                sentences = [sentence.replace("[PERIOD]", ".") for sentence in sentences]
                

                document_dict[folder_year_name]["months_data"][folder_month_name]["file_data"][file_name] = {
                    "content": plain_text,
                    "word_count": len(plain_text.split()),
                    "char_count": len(plain_text),
                    "sentence_count": len(sentences),
                    "sentences": sentences
                }

                document_dict[folder_year_name]["months_data"][folder_month_name]["sentence_count"] += len(sentences)
                document_dict[folder_year_name]["months_data"][folder_month_name]["word_count"] += len(plain_text.split())

                document_dict[folder_year_name]["sentence_count"] += len(sentences)
                document_dict[folder_year_name]["word_count"] += len(plain_text.split())
                

with open("data.json", "w", encoding="utf-8") as f:
    json.dump(document_dict, f, ensure_ascii=False, indent=4)


## Vizualization of data.

Average word count and sentence count per year.

In [7]:
# Load the JSON file
with open("data.json", "r", encoding="utf-8") as f:
    document_dict = json.load(f)

Convert to Dataframe.

In [ ]:
data = document_dict

records = []
for year in ["promet_2022", "promet_2023", "promet_2024"]:
    for month in document_dict[year]["months_data"].keys():
        month_data = document_dict[year]["months_data"][month]
        file_data_dict = month_data["file_data"]

# Create a list of flattened records

        for file_name, file_details in file_data_dict.items():
            record = {
                # File-specific fields
                "year": year,
                "month": month,
                "file_name": file_name,
                "content": file_details["content"],
                "file_word_count": file_details["word_count"],
                "file_char_count": file_details["char_count"],
                "file_sentence_count": file_details["sentence_count"],
                "sentences": file_details["sentences"],
                
                # Month-specific fields
                "month_files": month_data["files"],
                "month_sentence_count": month_data["sentence_count"],
                "month_word_count": month_data["word_count"],

                # Year-specific fields
                "months": document_dict[year]["months"],
                "year_sentence_count": document_dict[year]["sentence_count"],
                "year_word_count": document_dict[year]["word_count"]
            }
            records.append(record)

# Convert to DataFrame
df = pd.DataFrame(records)

# Display the DataFrame
df.head()

,year,month,file_name,content,file_word_count,file_char_count,file_sentence_count,sentences,month_files,month_sentence_count,month_word_count,months,year_sentence_count,year_word_count
0,promet_2022,april_2022,TMP-1.rtf,Prometne informacije 30. 04. 2022 18.30 1. in ...,74,431,4,[Prometne informacije 30. 04. 2022 18.30 1. in...,741,3832,55464,12,51490,740176
1,promet_2022,april_2022,TMP-10.rtf,Prometne informacije 30. 04. 2022 13.00 1. in ...,61,380,3,[Prometne informacije 30. 04. 2022 13.00 1. in...,741,3832,55464,12,51490,740176
2,promet_2022,april_2022,TMP-100.rtf,Prometne informacije 27. 04. 2022 6.30 1. prog...,47,337,3,[Prometne informacije 27. 04. 2022 6.30 1. pro...,741,3832,55464,12,51490,740176
3,promet_2022,april_2022,TMP-101.rtf,Prometne informacije 27. 04. 2022 6.00 1. in 2...,47,294,3,[Prometne informacije 27. 04. 2022 6.00 1. in ...,741,3832,55464,12,51490,740176
4,promet_2022,april_2022,TMP-102.rtf,Prometne informacije 26. 04. 2022 20.00 2. pro...,71,467,5,[Prometne informacije 26. 04. 2022 20.00 2. pr...,741,3832,55464,12,51490,740176


Average word count per file, month and year.